In [1]:
import pandas as pd

# Load the dataset (adjust file path as needed)
df = pd.read_csv('D:\\elte_res\\OneDrive - Eotvos Lorand Tudomanyegyetem\\CSE\\Spring 20205\\Data Security\\netflow_evasion\\data\\UNSW-NB15_1_partial_binarised.csv')


# --- 2. Most used 'sport' when label == 0 ---
most_common_sport_label_0 = df[df['label'] == 0]['sport'].mode()
print(f"\nMost used sport for label 0: {most_common_sport_label_0.iloc[0]}")

# --- 3. Average duration by label ---
avg_duration = df.groupby('label')['dur'].mean()
print("\nAverage duration by label:")
print(avg_duration)

# --- 4. Average number of spkts by label ---
avg_spkts = df.groupby('label')['spkts'].mean()
print("\nAverage source packets (spkts) by label:")
print(avg_spkts)

# --- 5. Average sbytes by label ---
avg_sbytes = df.groupby('label')['sbytes'].mean()
print("\nAverage source bytes (sbytes) by label:")
print(avg_sbytes)

#6 most spkts count when the label is 0

most_common_spkts_label_0 = df[df['label'] == 0]['spkts'].mode()
print(f"\nMost used spkts for label 0: {most_common_spkts_label_0.iloc[0]}")


# 7 zone transfer tcp connection trick

count = df[(df['dport'] == 53) & (df['proto'].str.lower() == 'tcp')].shape[0]

print(f"\nNumber of times dport == 53 and proto == 'tcp': {count}")



C:\Users\why_g\AppData\Local\Temp\ipykernel_30192\3969635554.py:4: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('D:\\elte_res\\OneDrive - Eotvos Lorand Tudomanyegyetem\\CSE\\Spring 20205\\Data Security\\netflow_evasion\\data\\UNSW-NB15_1_partial_binarised.csv')



Most used sport for label 0: 0

Average duration by label:
label
0    0.853138
1    1.176211
Name: dur, dtype: float64

Average source packets (spkts) by label:
label
0    42.541830
1    17.485258
Name: spkts, dtype: float64

Average source bytes (sbytes) by label:
label
0     4745.577972
1    13066.138330
Name: sbytes, dtype: float64

Most used spkts for label 0: 2

Number of times dport == 53 and proto == 'tcp': 0


In [2]:
import pandas as pd

# Load your CSV file
df = pd.read_csv('D:\\elte_res\\OneDrive - Eotvos Lorand Tudomanyegyetem\\CSE\\Spring 20205\\Data Security\\netflow_evasion\\data\\UNSW-NB15_1_partial_binarised.csv')

# Convert 'sport' values to integers, handling both decimal and hex formats
def parse_port(val):
    try:
        # If the value starts with '0x', interpret it as hexadecimal
        if isinstance(val, str) and val.lower().startswith('0x'):
            return int(val, 16)
        return int(val)  # otherwise, treat it as decimal
    except:
        return None  # return None for invalid values

# Apply parsing to the 'sport' column
df['sport_int'] = df['sport'].apply(parse_port)

# Drop rows with invalid sport values
df_valid = df.dropna(subset=['sport_int'])

# --- Identify unused ports in the range 0–65535 ---
all_ports = set(range(0, 65536))
used_ports = set(df_valid['sport_int'].unique())
unused_ports = all_ports - used_ports

print(f"\nNumber of unused source ports (0–65535): {len(unused_ports)}")
print(f"Some unused ports: {sorted(list(unused_ports))[-7:-1]}")


C:\Users\why_g\AppData\Local\Temp\ipykernel_30192\3576835674.py:4: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('D:\\elte_res\\OneDrive - Eotvos Lorand Tudomanyegyetem\\CSE\\Spring 20205\\Data Security\\netflow_evasion\\data\\UNSW-NB15_1_partial_binarised.csv')



Number of unused source ports (0–65535): 996
Some unused ports: [10880, 23544, 25940, 26345, 29230, 54585]


In [4]:
# First, ensure dport is numeric (handle hex if needed, like we did for sport)
def parse_port(val):
    try:
        if isinstance(val, str) and val.lower().startswith('0x'):
            return int(val, 16)
        return int(val)
    except:
        return None

df['dport_int'] = df['dport'].apply(parse_port)

# Filter rows where dport is 5060
df_5060 = df[df['dport_int'] == 5060]

# Group by label and calculate average sbytes
avg_sbytes_dport_5060 = df_5060.groupby('label')['sbytes'].mean()

print("\nAverage sbytes by label when dport == 5060:")
print(avg_sbytes_dport_5060)



Average sbytes by label when dport == 5060:
label
0    3463.333333
1    7551.659574
Name: sbytes, dtype: float64


In [5]:
# First, ensure dport is numeric (handle hex if needed, like we did for sport)
def parse_port(val):
    try:
        if isinstance(val, str) and val.lower().startswith('0x'):
            return int(val, 16)
        return int(val)
    except:
        return None

df['dport_int'] = df['dport'].apply(parse_port)

# Filter rows where dport is 80
df_80 = df[df['dport_int'] == 80]

# Group by label and calculate average sbytes
avg_sbytes_dport_80 = df_80.groupby('label')['sbytes'].mean()

print("\nAverage sbytes by label when dport == 80:")
print(avg_sbytes_dport_80)



Average sbytes by label when dport == 80:
label
0    3946.991407
1    8479.095255
Name: sbytes, dtype: float64


In [ ]:
# Filter for sport == 80 and label == 1
df_filtered = df[(df['dport'] == 25) ]

# Calculate average duration
avg_duration = df_filtered.groupby('label')['dur'].mean()

print(f"\nAverage duration when sport == 80 and label == 1: {avg_duration}")





Average duration when sport == 80 and label == 1: label
0    0.320582
Name: dur, dtype: float64


In [ ]:

##################################################Generating Data###############################################################################

import random
import csv

# === CONFIGURATION ===
original_record = {
    "sport": 11233,
    "dport": 25,
    "proto": "tcp",
    "state": "FIN",
    "dur": 1.547788,
    "sbytes": 65540,
    "spkts":16
}

# How many variants to generate
total_variants = 1000

# Output file
output_file = "evaded_variants.csv"

# === VARIANT GENERATOR ===
def generate_variants(base_record, n):
    variants = []
    for _ in range(n):
        variant = base_record.copy()
        # ±3 port variation
        variant["sport"] = base_record["sport"] + random.choice([-3, -2, -1, 0, 1, 2, 3])
        # ±1e-6 duration variation
        #variant["dur"] = base_record["dur"] + random.choice([-0.000003,-0.000002,-0.000001,0,0.000001,0.000002,0.000003])
        variant["dur"] = round(base_record["dur"] + random.choice([-2e-6, -1e-6, 0, 1e-6, 2e-6]), 8)

        # ±3 bytes
        variant["sbytes"] = base_record["sbytes"] + random.choice([-3, -2, -1, 0, 1, 2, 3])
        # spkts in 2–4
        variant["spkts"] = random.choice([16,17,18])
        variants.append(variant)
    return variants

# === WRITE TO CSV ===
def write_csv(records, filename):
    fieldnames = list(records[0].keys())
    with open(filename, mode='w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in records:
            writer.writerow(row)

# === MAIN ===
if __name__ == "__main__":
    evaded_records = generate_variants(original_record, total_variants)
    write_csv(evaded_records, output_file)
    print(f"✅ Generated {total_variants} evasion records in '{output_file}'")


✅ Generated 1000 evasion records in 'evaded_variants.csv'
